In [0]:
# passing a value using parameter #

# Create widgets
dbutils.widgets.text("source_system", "", "Source System")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("source_file_format", "", "Source File Format")

# Get widget values into variables
source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
# build landing path and check if new files were received #

landing_path = f'/Volumes/adb_caresync/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/'

try:
    landing_files = dbutils.fs.ls(landing_path)
    path_exists = len(landing_files) > 0
except Exception:
    # path or one of its parent directories does not exist
    path_exists = False

if not path_exists:
    print(f"No new files are received at {landing_path}. Exiting notebook.")
    dbutils.notebook.exit(f"No new files are received at {landing_path}")

In [0]:
#loading the parquet data #

if source_file_format == 'csv':
    landing_df = spark.read.format('csv').option('header', True).option('inferSchema', True).load(landing_path)
else:
    landing_df = spark.read.format('parquet').load(landing_path)

In [0]:
from pyspark.sql import functions as F

updated_df = landing_df.withColumn("landing_timestamp", F.lit(landing_timestamp)) \
                       .withColumn("insert_timestamp", F.current_timestamp())

In [0]:
#append into bronze table#

updated_df.write.mode("append").saveAsTable(f"adb_caresync.bronze.{table_name}")


In [0]:
%sql
-- select * from adb_caresync.bronze.hospitals
-- comment : write a sql query for testing purpose
